In [0]:
%pip install --quiet feedparser beautifulsoup4 httpx lxml curl_cffi
dbutils.library.restartPython()


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# =============================================================================
# Imports
# =============================================================================
import os
import re
import json
import time
import random
import hashlib
import unicodedata
from datetime import datetime, timezone, timedelta
from email.utils import parsedate_to_datetime
from typing import Optional

import feedparser
import httpx
from bs4 import BeautifulSoup

from curl_cffi import requests as cffi_requests


In [0]:
# =============================================================================
# Configuração — fonte lida via parâmetro do Job (widget), não fixa no código
# =============================================================================

dbutils.widgets.remove("fonte")
dbutils.widgets.text("fonte", "todas")
NOME_FONTE = dbutils.widgets.get("fonte")

# Registro central de fontes RSS. Adicionar fonte nova = adicionar entrada
# aqui, não criar notebook novo. "feeds" é sempre uma lista — a maioria tem
# 1 item, o Megawhat tem 2 (combinados com dedupe automático por URL).
CONFIGS_FONTES = {
    "creditoprivado360": {
        "feeds": ["https://creditoprivado360.com.br/feed/"],
        "source_id": "credito_privado_360",
        "source_descricao": "Linked from Crédito Privado 360",
    },
    "neofeed": {
        "feeds": ["https://neofeed.com.br/feed/"],
        "source_id": "neofeed",
        "source_descricao": "Linked from NeoFeed",
    },
    "megawhat": {
        "feeds": [
            "https://megawhat.uol.com.br/destaques-do-diario/feed",
            "https://megawhat.uol.com.br/ultimas-noticias/feed",
        ],
        "source_id": "megawhat",
        "source_descricao": "Linked from Megawhat",
    },
    "agencia_eixos": {
    "feeds": ["https://eixos.com.br/feed/"],
    "source_id": "agencia_eixos",
    "source_descricao": "Linked from Agência Eixos",
},
"agencia_infra": {
    "feeds": ["https://agenciainfra.com/blog/feed/"],
    "source_id": "agencia_infra",
    "source_descricao": "Linked from Agência iNFRA",
},
}

HOJE = datetime.now(timezone.utc).astimezone().strftime("%Y-%m-%d")

PASTA_DESTINO = f"/Volumes/desafio_kinea/research/research_volume/infraestrutura/files/{HOJE}"
os.makedirs(PASTA_DESTINO, exist_ok=True)
print(f"[setup] Salvando artefatos em: {PASTA_DESTINO}")

JANELA_HORAS = 24

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:126.0) Gecko/20100101 Firefox/126.0",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36 Edg/124.0.0.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 "
    "(KHTML, like Gecko) Version/17.4 Safari/605.1.15",
]

IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124", "safari17_0", "edge101"]

HTTP_TIMEOUT = 30

MIN_CHARS_TEXTO = 200


[setup] Salvando artefatos em: /Volumes/desafio_kinea/research/research_volume/infraestrutura/files/2026-08-01


In [0]:
# =============================================================================
# Helpers
# =============================================================================

def slugify(texto: str, max_len: int = 80) -> str:
    if not texto:
        return "sem-titulo"
    nfkd = unicodedata.normalize("NFKD", texto)
    ascii_txt = nfkd.encode("ascii", "ignore").decode("ascii")
    ascii_txt = re.sub(r"[^a-zA-Z0-9]+", "-", ascii_txt).strip("-").lower()
    return (ascii_txt[:max_len] or "sem-titulo").strip("-")


def hash_curto(texto: str, n: int = 8) -> str:
    return hashlib.md5(texto.encode("utf-8")).hexdigest()[:n]


def parsear_data_rss(data_str: str) -> str:
    try:
        return parsedate_to_datetime(data_str).strftime("%Y-%m-%d")
    except Exception:
        return HOJE


def parece_paywall(texto: str) -> bool:
    marcadores = [
        "para continuar lendo",
        "assine agora mesmo",
        "assine já",
        "conteúdo exclusivo para assinantes",
        "faça login para ler",
        "este conteúdo é para assinantes",
        "cadastre-se para continuar",
    ]
    trecho = texto[:1500].lower()
    return any(m in trecho for m in marcadores)


def headers_aleatorios(referer: Optional[str] = None) -> dict:
    ua = random.choice(USER_AGENTS)
    headers = {
        "User-Agent": ua,
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,"
                  "image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.8",
        "Accept-Encoding": "gzip, deflate",
        "Cache-Control": "no-cache",
        "Pragma": "no-cache",
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "none",
        "Sec-Fetch-User": "?1",
        "Upgrade-Insecure-Requests": "1",
    }
    if referer:
        headers["Referer"] = referer
    return headers


In [0]:
# =============================================================================
# Etapa 1 — Buscar e combinar os feeds da fonte selecionada
# =============================================================================

def buscar_itens_de_feed(url: str, tentativas: int = 2) -> list:
    """Busca um feed específico, com retry. Retorna lista de entries (vazia se falhar)."""
    for tentativa in range(1, tentativas + 1):
        try:
            resp = httpx.get(url, timeout=15, follow_redirects=True,
                              headers={"User-Agent": random.choice(USER_AGENTS)})
        except Exception as e:
            print(f"[feed] {url} -> tentativa {tentativa}/{tentativas} falhou: {e}")
            if tentativa < tentativas:
                time.sleep(random.uniform(1.0, 2.5))
            continue

        if resp.status_code != 200:
            print(f"[feed] {url} -> HTTP {resp.status_code}.")
            return []

        parsed = feedparser.parse(resp.content)
        print(f"[feed] OK: '{url}', {len(parsed.entries)} entries.")
        return parsed.entries

    print(f"[feed] {url} -> todas as tentativas falharam, seguindo sem esse feed.")
    return []


def combinar_feeds(urls: list) -> list:
    """Busca todos os feeds da lista e junta os itens, removendo duplicata por URL."""
    vistos = set()
    combinados = []

    for url in urls:
        for entry in buscar_itens_de_feed(url):
            link = entry.get("link")
            if link and link in vistos:
                continue
            if link:
                vistos.add(link)
            combinados.append(entry)

    print(f"[feed] Total combinado: {len(combinados)} itens únicos de {len(urls)} feed(s).")
    return combinados


def dentro_da_janela(entry, horas: int = JANELA_HORAS) -> bool:
    if not entry.get("published_parsed"):
        print(f"    -> aviso: item sem published_parsed ({entry.get('title', '?')[:60]}); incluindo mesmo assim.")
        return True

    pub_dt = datetime(*entry.published_parsed[:6], tzinfo=timezone.utc)
    limite = datetime.now(timezone.utc) - timedelta(hours=horas)
    return pub_dt >= limite


def selecionar_itens(itens: list) -> list:
    antes = len(itens)
    itens = [e for e in itens if dentro_da_janela(e)]
    print(f"[filtro] janela de {JANELA_HORAS}h: {antes} itens -> {len(itens)} dentro do prazo.")
    return itens


In [0]:
# =============================================================================
# Etapa 2 — Download do HTML da matéria
# =============================================================================

def baixar_html(url: str) -> Optional[str]:
    headers = headers_aleatorios()

    time.sleep(random.uniform(0.5, 1.5))

    try:
        resp = httpx.get(url, headers=headers, timeout=20, follow_redirects=True)
        if resp.status_code == 200 and resp.text:
            return resp.text
        print(f"    -> httpx retornou HTTP {resp.status_code}; tentando fallback curl_cffi.")
    except Exception as e:
        print(f"    -> httpx falhou ({e}); tentando fallback curl_cffi.")

    try:
        resp = cffi_requests.get(url, headers=headers, impersonate=random.choice(IMPERSONATE_PROFILES), timeout=20)
        if resp.status_code == 200 and resp.text:
            return resp.text
        print(f"    -> curl_cffi também retornou HTTP {resp.status_code}.")
    except Exception as e:
        print(f"    -> curl_cffi também falhou: {e}")

    return None


In [0]:
# =============================================================================
# Etapa 3 — Limpar o HTML e extrair só o texto útil
# =============================================================================

TAGS_LIXO = [
    "script", "style", "noscript", "iframe", "svg", "form",
    "nav", "footer", "header", "aside", "button",
]

def extrair_texto(html: str) -> str:
    if not html:
        return ""

    try:
        soup = BeautifulSoup(html, "lxml")
    except Exception as e:
        print(f"    -> lxml falhou ao parsear ({e}); usando html.parser como fallback.")
        soup = BeautifulSoup(html, "html.parser")

    for tag in soup(TAGS_LIXO):
        tag.decompose()

    article = soup.find("article")
    if article and len(article.get_text(strip=True)) > 500:
        base = article
    else:
        base = soup

    texto = base.get_text("\n", strip=True)
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    return texto.strip()


In [0]:
# =============================================================================
# Etapa 4 — Salvar no Volume
# =============================================================================

def salvar_artefatos(pasta: str, titulo: str, texto: str, metadados: dict) -> tuple[str, str]:
    slug_source = slugify(SOURCE_ID, max_len=40)
    slug_titulo = slugify(titulo, max_len=60) or "sem-titulo"
    sufixo_hash = hash_curto(metadados.get("url") or titulo)

    nome_base = f"{slug_source}_{slug_titulo}_{sufixo_hash}"
    caminho_txt = os.path.join(pasta, f"{nome_base}.txt")
    caminho_json = os.path.join(pasta, f"{nome_base}.json")

    with open(caminho_txt, "w", encoding="utf-8") as f:
        f.write(texto or "")

    with open(caminho_json, "w", encoding="utf-8") as f:
        json.dump(metadados, f, ensure_ascii=False, indent=2)

    return caminho_txt, caminho_json


In [0]:
# =============================================================================
# Etapa 5 — Pipeline principal
# =============================================================================

def processar_entry(entry) -> Optional[dict]:
    titulo = entry.get("title", "sem-titulo")
    url = entry.get("link")
    print(f"\n  [item] {titulo[:100]}")

    if not url:
        print("    -> sem link; pulando.")
        return None

    html = baixar_html(url)
    if not html:
        print("    -> download do HTML falhou; pulando.")
        return None

    texto = extrair_texto(html)

    if parece_paywall(texto):
        print("    -> marcador de paywall detectado; pulando.")
        return None

    if not texto or len(texto) < MIN_CHARS_TEXTO:
        print(f"    -> texto muito curto ({len(texto)} chars); pulando.")
        return None

    data_publicacao = parsear_data_rss(entry.get("published", ""))

    metadados = {
        "source_id": SOURCE_ID,
        "title": titulo,
        "description": SOURCE_DESCRICAO,
        "url": url,
        "date": HOJE,
        "published_at": data_publicacao,
    }

    caminho_txt, caminho_json = salvar_artefatos(PASTA_DESTINO, titulo, texto, metadados)
    print(f"    -> salvo em {caminho_txt}")

    return {"titulo": titulo, "url": url, "caminho_txt": caminho_txt, "caminho_json": caminho_json}


In [0]:
# =============================================================================
# Execução — roda todas as fontes de CONFIGS_FONTES em sequência
# (ou só uma, se "fonte" for passado com um nome específico em vez de "todas")
# =============================================================================

if NOME_FONTE == "todas":
    fontes_a_rodar = CONFIGS_FONTES
else:
    if NOME_FONTE not in CONFIGS_FONTES:
        raise ValueError(f"Fonte {NOME_FONTE!r} não configurada. Opções: {list(CONFIGS_FONTES)}")
    fontes_a_rodar = {NOME_FONTE: CONFIGS_FONTES[NOME_FONTE]}

resumo_geral = {}

for nome_fonte, config in fontes_a_rodar.items():
    print(f"\n{'='*70}")
    print(f"=== Processando fonte: {nome_fonte!r} ===")
    print(f"{'='*70}")

    # Redefine as variáveis que os helpers (processar_entry, salvar_artefatos)
    # já leem — não precisa mudar nenhuma outra célula do notebook.
    FEEDS_A_COMBINAR = config["feeds"]
    SOURCE_ID = config["source_id"]
    SOURCE_DESCRICAO = config["source_descricao"]

    try:
        itens_brutos = combinar_feeds(FEEDS_A_COMBINAR)
        itens = selecionar_itens(itens_brutos)
        print(f"=== {len(itens)} itens a processar ===")

        resultados_fonte = []
        for entry in itens:
            try:
                resultado = processar_entry(entry)
                if resultado:
                    resultados_fonte.append(resultado)
            except Exception as e:
                print(f"[ERRO] item {entry.get('title', '?')!r} falhou: {e}")

        resumo_geral[nome_fonte] = len(resultados_fonte)
        print(f"=== Fim de {nome_fonte!r}: {len(resultados_fonte)} matérias salvas ===")

    except Exception as e:
        print(f"[ERRO GERAL] fonte {nome_fonte!r} falhou por completo: {e}")
        resumo_geral[nome_fonte] = f"ERRO: {e}"

print(f"\n\n{'='*70}")
print("=== RESUMO FINAL ===")
for nome_fonte, resultado in resumo_geral.items():
    print(f"  {nome_fonte}: {resultado}")
print(f"{'='*70}")


=== Processando fonte: 'creditoprivado360' ===
[feed] OK: 'https://creditoprivado360.com.br/feed/', 10 entries.
[feed] Total combinado: 10 itens únicos de 1 feed(s).
[filtro] janela de 24h: 10 itens -> 0 dentro do prazo.
=== 0 itens a processar ===
=== Fim de 'creditoprivado360': 0 matérias salvas ===

=== Processando fonte: 'neofeed' ===
[feed] OK: 'https://neofeed.com.br/feed/', 10 entries.
[feed] Total combinado: 10 itens únicos de 1 feed(s).
[filtro] janela de 24h: 10 itens -> 5 dentro do prazo.
=== 5 itens a processar ===

  [item] De Salerno para a Serra da Mantiqueira: como Dani Branca virou o melhor pizzaiolo do Brasil
    -> salvo em /Volumes/desafio_kinea/research/research_volume/infraestrutura/files/2026-08-01/neofeed_de-salerno-para-a-serra-da-mantiqueira-como-dani-branca-viro_61999c49.txt

  [item] O fantasma do “goût de fumée” ronda a safra de vinhos franceses
    -> salvo em /Volumes/desafio_kinea/research/research_volume/infraestrutura/files/2026-08-01/neofeed_o-fantas